# Classifying Commuter vs. Recreational Citi Bike Trips
### What Trip Features Predict Ride Purpose in New York City?

**Programming Tools for Urban Analytics** | Dr Qunshan Zhao  
**By:** Fabian Moreno

---

### Setup

In [8]:
#!pip install shap
#!pip install dotenv
#!pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.2 MB/s  0:00:00 eta 0:00:01


In [ ]:
#  ----- Standard library 
import os
import sys
from pathlib import Path


# ----- Adding project root to the path
sys.path.insert(0, str(Path('..').resolve()))


# ----- Libraries for data manipualtion
import numpy as np
import pandas as pd
import duckdb

# ----- Libraries for APIs
import requests


# ----- Spatial libraries
import geopandas as gpd
import folium
from shapely.geometry import Point


# ----- Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')


# ----- libraries for machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import shap


# ----- Utility libraries
from dotenv import load_dotenv
from tqdm import tqdm
import holidays


# ----- Project modules
from src.utils import DATA_RAW, DATA_PRO, DB_PATH, log
load_dotenv()


# ----- DuckDB connection
con = duckdb.connect(str(DB_PATH))

print(f"DuckDB connected: {DB_PATH}")
print(f"DuckDB {duckdb.__version__}")

DuckDB connected: /Users/fabianmorenogarza/Documents/UrbanAnalytics/01_PTUA/citibike/data/citibike.duckdb
DuckDB 1.4.3


---
## Introduction

*(~400 words)*

**Research question:** *What trip-level features best predict whether a Citi Bike journey is made for commuting or recreational purposes, and what does the spatial and temporal distribution of classified trips reveal about urban mobility patterns in New York City?*

---
## Data Collection

*(~350 words)*

### 2.1 Download citibike data from S3

In [ ]:
from src.data_download import download_citibike_trips

# Download full year 2024 — skips January since it's already on disk
trip_files = download_citibike_trips(year=2024)

print({len(trip_files)})

In [ ]:
# Verifying all files were downloaded correctly

files = sorted(DATA_RAW.glob("*-citibike-tripdata.parquet"))
total_rows = 0

for f in files:
    df = pd.read_parquet(f)
    total_rows += len(df)
    print(f"{f.name}: {len(df):,} rows")

In [18]:
# Quick look of the downloaded data
df_sample = pd.read_parquet(DATA_RAW / "202406-citibike-tripdata.parquet")
print(df_sample.shape)
print(df_sample.dtypes)
df_sample.head(3)

(4783576, 13)
ride_id                          str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
dtype: object


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,38F52721E98C6761,electric_bike,2024-06-19 19:02:23.487,2024-06-19 19:39:04.984,Pier 61 at Chelsea Piers,6233.04,Peck Slip & South St,5096.12,40.746872,-74.008210,40.707519,-74.001081,member
1,C8A3A961E2F7BB14,electric_bike,2024-06-25 07:52:20.777,2024-06-25 07:59:22.760,41 Ave & 77 St,6290.06,Tyler Ave & Maurice Ave,5881.01,40.745570,-73.888200,40.733940,-73.901280,member
2,90690D5942906B7C,classic_bike,2024-06-15 07:32:02.957,2024-06-15 07:33:32.205,Prospect Park SW & 10 Ave,3611.02,Windsor Pl & Howard Pl,3579.04,40.659945,-73.977504,40.659491,-73.980139,member


### 2.2 GBFS station metadata

In [21]:
from src.data_download import fetch_gbfs_stations

stations = fetch_gbfs_stations()
print(f"{len(stations)} stations")
print(stations.dtypes)
stations.head(3)

[00:54:50] [2.2] stations.parquet already on disk — loading
2360 stations
station_id        str
name              str
lat           float64
lon           float64
capacity        int64
region_id         str
dtype: object


,station_id,name,lat,lon,capacity,region_id
0,1905837242740508940,31 St & Broadway,40.762110,-73.925230,35,71
1,41495491-5d89-4e14-aab9-c3db04aad399,43 St & Skillman Ave,40.746927,-73.920825,19,71
2,2177014969129222184,Henry Hudson Pkwy E & W 231 St,40.883360,-73.915050,0,71


### 2.3 OpenWeatherMap API

1. Sign up at openweathermap.org/api
2. Add the key to your .env file when it arrives
3. Run fetch_weather_history()

### 2.4 NYC PLUTO land use data

In [23]:
import requests, io, pandas as pd


# print columns from NYC Pluto
r = requests.get("https://data.cityofnewyork.us/api/views/64uk-42ks/rows.csv?accessType=DOWNLOAD", timeout=180)
cols = pd.read_csv(io.StringIO(r.content.decode("utf-8")), nrows=0).columns.tolist()
print(cols)

['borough', 'Tax block', 'Tax lot', 'community board', 'census tract 2010', 'cb2010', 'schooldist', 'council district', 'postcode', 'firecomp', 'policeprct', 'healtharea', 'sanitboro', 'sanitsub', 'address', 'zonedist1', 'zonedist2', 'zonedist3', 'zonedist4', 'overlay1', 'overlay2', 'spdist1', 'spdist2', 'spdist3', 'ltdheight', 'splitzone', 'bldgclass', 'landuse', 'easements', 'ownertype', 'ownername', 'lotarea', 'bldgarea', 'comarea', 'resarea', 'officearea', 'retailarea', 'garagearea', 'strgearea', 'factryarea', 'otherarea', 'areasource', 'numbldgs', 'numfloors', 'unitsres', 'unitstotal', 'lotfront', 'lotdepth', 'bldgfront', 'bldgdepth', 'ext', 'proxcode', 'irrlotcode', 'lottype', 'bsmtcode', 'assessland', 'assesstot', 'exempttot', 'yearbuilt', 'yearalter1', 'yearalter2', 'histdist', 'landmark', 'builtfar', 'residfar', 'commfar', 'facilfar', 'borocode', 'BBL', 'condono', 'tract2010', 'xcoord', 'ycoord', 'latitude', 'longitude', 'zonemap', 'zmcode', 'sanborn', 'taxmap', 'edesignum', '

#### PLUTO columns to be kept

| Column | Why we need it |
|--------|----------------|
| `BBL` | Unique lot ID: identifies each tax lot |
| `landuse` | The key one: codes like `05` = commercial, `09` = park, `01` = residential. Used to tag stations as employment-anchored or recreational |
| `bldgclass` | Building type: extra detail on top of landuse |
| `zonedist1` | Zoning district: commercial / residential / mixed |
| `xcoord` / `ycoord` | NYC grid coordinates |
| `latitude` / `longitude` | Standard coordinates: used to spatially join stations to surrounding lots |
| `bct2020` | Census tract |
| `borough` | Manhattan, Brooklyn, Queens, Bronx, Staten Island |

In [24]:
# Create a list for columns to be kept
keep_cols = [
    "BBL", "landuse", "bldgclass", "zonedist1",
    "xcoord", "ycoord", "latitude", "longitude",
    "bct2020", "borough",
]

df = df.dropna(subset=["latitude", "longitude"])

In [27]:
from importlib import reload
import src.data_download
reload(src.data_download)
from src.data_download import download_pluto

pluto = download_pluto(overwrite=True)
print(f"{len(pluto):,} lots")
print(pluto.dtypes)
pluto.head(3)

[14:38:42] [2.4] Downloading NYC PLUTO from NYC Open Data (~300 MB)...
[14:42:12] [2.4] Saved 857,161 PLUTO lots → pluto.parquet
857,161 lots
borough      str
zonedist1    str
bldgclass    str
landuse      str
BBL          str
xcoord       str
ycoord       str
latitude     str
longitude    str
bct2020      str
dtype: object


,borough,zonedist1,bldgclass,landuse,BBL,xcoord,ycoord,latitude,longitude,bct2020
0,QN,R3X,A5,1,4061730023,1047384,219559,40.7690896,-73.7720738,4112300
1,QN,R3X,A0,1,4061730024,1047363,219486,40.7688894,-73.7721503,4112300
2,QN,R2A,B3,1,4061690026,1046855,219159,40.7679955,-73.7739873,4112300


latitude, longitude, xcoord, and ycoord are still str instead of float64

In [29]:
# ----- 

# Fix coordinate dtypes
for col in ("latitude", "longitude", "xcoord", "ycoord"):
    pluto[col] = pd.to_numeric(pluto[col], errors="coerce")

pluto = pluto.dropna(subset=["latitude", "longitude"])

# Save fixed version
pluto.to_parquet(DATA_RAW / "pluto.parquet", index=False)

print(f"{len(pluto):,} lots")
print(pluto.dtypes)

857,161 lots
borough          str
zonedist1        str
bldgclass        str
landuse          str
BBL              str
xcoord         int64
ycoord         int64
latitude     float64
longitude    float64
bct2020          str
dtype: object


In [ ]:
from importlib import reload
import src.database
reload(src.database)
from src.database import create_schema, ingest_stations, ingest_trips, ingest_pluto, validate

create_schema(con)
ingest_stations(con)
ingest_trips(con)
ingest_pluto(con)
validate(con)

[15:01:27] [3.1] Creating DuckDB schema...
[15:01:27] [3.1] Schema created — 4 tables ready
[15:01:27] [3.2] Ingesting stations...
[15:01:27] [3.2] stations: 2,360 rows
[15:01:27] [3.2] Ingesting trips (this may take a minute)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[15:01:37] [3.2] trips: 44,303,209 rows
[15:01:37] [3.2] Ingesting PLUTO...


ConversionException: Conversion Error: Could not convert string '4061730023' to INT32 when casting from source column BBL